In [2]:
# 데이터 구조 확인 단계: 노트북이 프로젝트 가상환경의 Python을 쓰는지 확인합니다.
import sys
print(sys.executable)

/Users/u102/workspace_public/021_Data_Analytics_Mini_Projcet/Data_Analytics_MiniProject_Skala/.venv/bin/python


In [3]:
# Batch1의 HDF5 참조를 읽고 표로 정리하는 데 필요한 도구를 불러옵니다.
from pathlib import Path

import h5py
import numpy as np
import pandas as pd

In [4]:
# 노트북을 notebooks/에서 실행하므로 원본 데이터는 ../data에 있습니다.
DATA_DIR = Path("../data")

batch1_path = DATA_DIR / "Batch1.mat"
batch2_path = DATA_DIR / "Batch2.mat"
batch3_path = DATA_DIR / "Batch3.mat"
extra_path = DATA_DIR / "extra.mat"

In [5]:
# 파일이 준비됐는지 먼저 확인해 뒤의 파일 열기 오류를 구분합니다.
for path in [batch1_path, batch2_path, batch3_path, extra_path]:
    print(path.name, path.exists())

Batch1.mat True
Batch2.mat True
Batch3.mat True
extra.mat True


In [6]:
# MATLAB v7.3 파일은 HDF5 형식이므로 h5py로 최상위 구조를 확인합니다.
with h5py.File(batch1_path, "r") as f:
    print(list(f.keys()))

['#refs#', '#subsystem#', 'batch', 'batch_date']


In [7]:
# batch의 필드별 크기와 자료형을 살펴 셀 수와 참조 저장 방식을 파악합니다.
with h5py.File(batch1_path, "r") as f:
    batch = f["batch"]

    for key in batch.keys():
        obj = batch[key]
        print(
            f"{key:15} "
            f"type={type(obj).__name__:10} "
            f"shape={obj.shape} "
            f"dtype={obj.dtype}"
        )

Vdlin           type=Dataset    shape=(46, 1) dtype=object
barcode         type=Dataset    shape=(46, 1) dtype=object
channel_id      type=Dataset    shape=(46, 1) dtype=object
cycle_life      type=Dataset    shape=(46, 1) dtype=object
cycles          type=Dataset    shape=(46, 1) dtype=object
policy          type=Dataset    shape=(46, 1) dtype=object
policy_readable type=Dataset    shape=(46, 1) dtype=object
summary         type=Dataset    shape=(46, 1) dtype=object


In [8]:
# cycle_life 한 건을 역참조해 실제 값이 참조 뒤에 저장됐음을 확인합니다.
with h5py.File(batch1_path, "r") as f:
    batch = f["batch"]

    ref = batch["cycle_life"][0, 0]
    print(ref)
    print(type(ref))

    obj = f[ref]
    print(type(obj))
    print(obj.shape)
    print(obj[()])

<HDF5 object reference>
<class 'h5py.h5r.Reference'>
<class 'h5py._hl.dataset.Dataset'>
(1, 1)
[[1190.]]


In [9]:
# 여러 셀에 반복 사용할 참조 해제 함수입니다. 숫자, MATLAB 문자, 배열을 구분합니다.
def read_h5_value(f, ref):
    """
    HDF5 object reference를 실제 값으로 변환
    """
    obj = f[ref]
    value = obj[()]

    # 숫자 1개짜리 배열
    if value.size == 1:
        return np.squeeze(value).item()

    # MATLAB 문자열은 uint16 배열로 저장되는 경우가 많음
    if np.issubdtype(value.dtype, np.integer):
        try:
            return "".join(chr(c) for c in value.flatten() if c != 0)
        except:
            pass

    return np.squeeze(value)

In [10]:
# EDA에 필요한 셀 단위 필드만 골라 각 참조의 실제 값을 추출합니다.
fields = [
    "barcode",
    "channel_id",
    "policy",
    "policy_readable",
    "cycle_life",
]

data = {}

with h5py.File(batch1_path, "r") as f:
    batch = f["batch"]

    for field in fields:
        values = []

        for ref in batch[field][:, 0]:
            values.append(read_h5_value(f, ref))

        data[field] = values

In [11]:
# 셀별 값으로 표를 만들고, Batch1 내부 순서인 cell_id와 정수형 수명을 추가합니다.
df_cells = pd.DataFrame(data)
df_cells["cycle_life"] = df_cells["cycle_life"].astype(int)

df_cells.insert(
    0,
    "cell_id",
    range(1, len(df_cells) + 1)
)

df_cells.head()

,cell_id,barcode,channel_id,policy,policy_readable,cycle_life
0,1,"[3707764736, 2, 1, 1, 1, 1]","[3707764736, 2, 1, 1, 47, 1]",3_6C-80PER_3_6C,3.6C(80%)-3.6C,1190
1,2,"[3707764736, 2, 1, 1, 2, 1]","[3707764736, 2, 1, 1, 48, 1]",3_6C-80PER_3_6C,3.6C(80%)-3.6C,1179
2,3,"[3707764736, 2, 1, 1, 3, 1]","[3707764736, 2, 1, 1, 49, 1]",3_6C-80PER_3_6C,3.6C(80%)-3.6C,1177
3,4,"[3707764736, 2, 1, 1, 4, 1]","[3707764736, 2, 1, 1, 50, 1]",4C-80PER_4C,4C(80%)-4C,1226
4,5,"[3707764736, 2, 1, 1, 5, 1]","[3707764736, 2, 1, 1, 51, 1]",4C-80PER_4C,4C(80%)-4C,1227


In [12]:
# channel_id의 여섯 요소를 분리해 Batch1에서 셀마다 달라지는 위치를 찾습니다.
channel_df = pd.DataFrame(
    df_cells["channel_id"].tolist(),
    columns=[
        "channel_0",
        "channel_1",
        "channel_2",
        "channel_3",
        "channel_4",
        "channel_5",
    ]
)

channel_df.head()

for col in channel_df.columns:
    print(
        col,
        "nunique =", channel_df[col].nunique(),
        "values =", channel_df[col].unique()[:10]
    )

channel_0 nunique = 1 values = [3707764736]
channel_1 nunique = 1 values = [2]
channel_2 nunique = 1 values = [1]
channel_3 nunique = 1 values = [1]
channel_4 nunique = 46 values = [47 48 49 50 51 52 53 54 55 56]
channel_5 nunique = 1 values = [1]


In [13]:
# 확인한 channel_4를 채널로 쓰고 EDA에 필요한 네 컬럼만 남깁니다.
df_cells["channel"] = channel_df["channel_4"]

df_eda = df_cells[
    [
        "cell_id",
        "channel",
        "policy_readable",
        "cycle_life"
    ]
].copy()

df_eda = df_eda.rename(
    columns={
        "policy_readable": "policy"
    }
)

df_eda["cycle_life"] = df_eda["cycle_life"].astype(int)

df_eda.head()

,cell_id,channel,policy,cycle_life
0,1,47,3.6C(80%)-3.6C,1190
1,2,48,3.6C(80%)-3.6C,1179
2,3,49,3.6C(80%)-3.6C,1177
3,4,50,4C(80%)-4C,1226
4,5,51,4C(80%)-4C,1227


In [14]:
# 다음 EDA 단계에서 재사용할 수 있도록 정리한 Batch1 셀 데이터를 CSV로 저장합니다.
from pathlib import Path

PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

df_eda.to_csv(
    PROCESSED_DIR / "batch1_cells.csv",
    index=False
)

In [15]:
with h5py.File(batch1_path, "r") as f:
    batch = f["batch"]

    summary_ref = batch["summary"][0, 0]
    summary = f[summary_ref]

    print(type(summary))
    print(list(summary.keys()))

<class 'h5py._hl.group.Group'>
['IR', 'QCharge', 'QDischarge', 'Tavg', 'Tmax', 'Tmin', 'chargetime', 'cycle']
